# Task 1: Parquet Internals and Metadata Inspection


## Step 1: Create a dataset with 500,000 rows using the following schema (you will reuse this dataset throughout the lab). Use np.random.seed(42) for reproducibility.



In [299]:
import numpy as np
import pandas as pd

np.random.seed(42)

data = pd.DataFrame({
    'user_id': np.arange(1, 500001),
    'city': np.random.choice(
        ['Berlin','Tokyo','New York','Dubai','Baku','Tbilisi','Paris','London','Moscow','Brazil'], 
        500000
    ),
    'score': np.random.uniform(0,100,500000),
    'active': np.random.choice([True, False], 500000),
    'signup_date': pd.to_datetime('today') - pd.to_timedelta(
        np.random.randint(0,365*3,500000), unit='D'
    ),
    'age': np.random.randint(18,81,500000),
    'sessions': np.random.randint(0,501,500000),
    'revenue': np.random.uniform(0,1000,500000)
})

data

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Paris,43.689490,False,2026-02-01 21:00:35.978702,71,323,179.164861
1,2,Dubai,97.403462,True,2025-06-02 21:00:35.978702,39,352,614.893856
2,3,London,66.148238,False,2025-11-08 21:00:35.978702,23,279,217.069799
3,4,Baku,21.993544,False,2025-05-22 21:00:35.978702,20,129,195.459273
4,5,Paris,91.719953,True,2023-04-02 21:00:35.978702,67,199,617.349590
...,...,...,...,...,...,...,...,...
499995,499996,Dubai,61.531318,False,2025-04-19 21:00:35.978702,67,448,157.442594
499996,499997,Berlin,74.712924,False,2025-10-25 21:00:35.978702,74,36,418.119068
499997,499998,Moscow,48.430697,True,2023-09-03 21:00:35.978702,27,205,803.122069
499998,499999,London,89.680757,True,2026-01-24 21:00:35.978702,50,432,602.050097


## Step 2: Save the dataset as a Parquet file. Then use pyarrow.parquet.ParquetFile to inspect:

The number of row groups
The number of rows and columns
The schema (column names and types)
For each column in the first row group: physical type, compressed size, and any available statistics (min, max, null count)

In [300]:
data.to_parquet('data.parquet', index= False)

In [301]:
import pyarrow.parquet as pq
import pyarrow as pa

In [302]:
parquet_file = pq.ParquetFile('data.parquet')

print(f"""The number of row groups: {parquet_file.metadata.num_row_groups}\n
The number of columns: {parquet_file.metadata.num_columns}\n
The number of rows: {parquet_file.metadata.num_rows}\n
Schema:\n{parquet_file.schema_arrow}""")



The number of row groups: 1

The number of columns: 8

The number of rows: 500000

Schema:
user_id: int64
city: large_string
score: double
active: bool
signup_date: timestamp[us]
age: int64
sessions: int64
revenue: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 988


In [303]:
rg = parquet_file.metadata.row_group(0)

for i in range(rg.num_columns):
    col = rg.column(i)
    stats = col.statistics

    print(f"{col.path_in_schema}")
    print("physical type:", col.physical_type)
    print("compressed size:", col.total_compressed_size)

    if stats:
        print("min:", stats.min)
        print("max:", stats.max)
        print("null count:", stats.null_count)

    print()

user_id
physical type: INT64
compressed size: 2273849
min: 1
max: 500000
null count: 0

city
physical type: BYTE_ARRAY
compressed size: 252513
min: Baku
max: Tokyo
null count: 0

score
physical type: DOUBLE
compressed size: 4272959
min: 5.188445665327279e-05
max: 99.99983148609545
null count: 0

active
physical type: BOOLEAN
compressed size: 63800
min: False
max: True
null count: 0

signup_date
physical type: INT64
compressed size: 698257
min: 2023-03-07 21:00:35.978702
max: 2026-03-05 21:00:35.978702
null count: 0

age
physical type: INT64
compressed size: 378360
min: 18
max: 80
null count: 0

sessions
physical type: INT64
compressed size: 567630
min: 0
max: 500
null count: 0

revenue
physical type: DOUBLE
compressed size: 4272959
min: 0.0009958058022618843
max: 999.9978869129644
null count: 0



## Step 3: Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.

In [304]:
import os

# save dataset as CSV
data.to_csv("data.csv", index=False)

csv_size = os.path.getsize("data.csv") / 1024
parquet_size = os.path.getsize("data.parquet") / 1024

compression_ratio = csv_size / parquet_size

print(f"""CSV size: {csv_size:.2f} KB
Parquet size: {parquet_size:.2f} KB
Compression ratio (CSV / Parquet): {compression_ratio:.2f}x""")

CSV size: 43556.47 KB
Parquet size: 12485.03 KB
Compression ratio (CSV / Parquet): 3.49x


## Step 4: Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, and why that matters for query performance

Parquet metadata stores important information such as the schema (column names and data types), row group structure, compression details, and column statistics like minimum, maximum, and null counts. CSV files do not contain this metadata; they only store raw text values.

This metadata allows query engines to optimize data access. For example, they can skip entire row groups if the statistics show that the required values are not present. Because CSV lacks this information, systems usually have to scan the entire file, which makes queries slower.

# Task 2: Column Projection and Selective Reads


## Step 1: Read the full Parquet file from Task 1 and time it.


In [305]:
%time pd.read_parquet('data.parquet')

CPU times: user 74.6 ms, sys: 48.3 ms, total: 123 ms
Wall time: 48.8 ms


,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Paris,43.689490,False,2026-02-01 21:00:35.978702,71,323,179.164861
1,2,Dubai,97.403462,True,2025-06-02 21:00:35.978702,39,352,614.893856
2,3,London,66.148238,False,2025-11-08 21:00:35.978702,23,279,217.069799
3,4,Baku,21.993544,False,2025-05-22 21:00:35.978702,20,129,195.459273
4,5,Paris,91.719953,True,2023-04-02 21:00:35.978702,67,199,617.349590
...,...,...,...,...,...,...,...,...
499995,499996,Dubai,61.531318,False,2025-04-19 21:00:35.978702,67,448,157.442594
499996,499997,Berlin,74.712924,False,2025-10-25 21:00:35.978702,74,36,418.119068
499997,499998,Moscow,48.430697,True,2023-09-03 21:00:35.978702,27,205,803.122069
499998,499999,London,89.680757,True,2026-01-24 21:00:35.978702,50,432,602.050097


## Step 2: Read only 2 columns from the same Parquet file and time it.


In [306]:
%time pd.read_parquet('data.parquet', columns=['city', 'score'])

CPU times: user 30.7 ms, sys: 11.9 ms, total: 42.6 ms
Wall time: 37.5 ms


,city,score
0,Paris,43.689490
1,Dubai,97.403462
2,London,66.148238
3,Baku,21.993544
4,Paris,91.719953
...,...,...
499995,Dubai,61.531318
499996,Berlin,74.712924
499997,Moscow,48.430697
499998,London,89.680757


## Step 3: Read only 2 columns from the CSV file and time it (you will need to read the entire CSV and select columns after, which is what pandas does).



In [307]:
%%time
pd.read_csv('data.csv')[['city', 'score']]


CPU times: user 646 ms, sys: 61.3 ms, total: 708 ms
Wall time: 719 ms


,city,score
0,Paris,43.689490
1,Dubai,97.403462
2,London,66.148238
3,Baku,21.993544
4,Paris,91.719953
...,...,...
499995,Dubai,61.531318
499996,Berlin,74.712924
499997,Moscow,48.430697
499998,London,89.680757


## Step 4: Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.

Parquet selective reads are faster because Parquet stores data in a column-oriented layout. Each column is stored separately inside row groups as column chunks. When a query requests only specific columns, the system can read only the relevant column chunks instead of scanning the entire dataset.

In Task 1, the metadata showed that each row group contains separate column chunks for every column. This structure allows Parquet to skip unnecessary columns entirely. In contrast, CSV stores data row by row, so the whole file must be read even if only a few columns are needed, which makes queries slower.

# Task 3: Predicate Pushdown in Practice


## Step 1: Using PyArrow, read the Parquet file with a filter (e.g., age > 50). Time the read.


In [308]:
import pyarrow.parquet as pq
import pyarrow as pa

In [309]:
%time pq.read_table('data.parquet', filters=[('age','>',50)])

CPU times: user 85.9 ms, sys: 23.6 ms, total: 110 ms
Wall time: 65.3 ms


pyarrow.Table
user_id: int64
city: large_string
score: double
active: bool
signup_date: timestamp[us]
age: int64
sessions: int64
revenue: double
----
user_id: [[1,5,6,7,8,...,131059,131067,131068,131071,131072],[131075,131078,131080,131081,131086,...,262137,262139,262140,262141,262144],[262148,262150,262151,262152,262155,...,393204,393206,393208,393212,393213],[393217,393218,393219,393220,393222,...,499991,499992,499996,499997,500000]]
city: [["Paris","Paris","Brazil","New York","Paris",...,"New York","Tbilisi","Tokyo","New York","Tokyo"],["Moscow","Baku","Brazil","Baku","Berlin",...,"Moscow","Berlin","London","Dubai","Tbilisi"],["Tbilisi","Tbilisi","London","London","Dubai",...,"New York","Tbilisi","Tbilisi","Tokyo","New York"],["New York","Baku","Baku","Tokyo","New York",...,"Dubai","Moscow","Dubai","Berlin","London"]]
score: [[43.689490033566045,91.71995305366835,60.49715376221981,41.78996051330323,27.220311643727456,...,69.69202532730313,20.260574805237873,73.70424820759605,46.3266

## Step 2: Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach

In [310]:
%%time

df = pd.read_parquet('data.parquet')
df[df['age'] > 50]

CPU times: user 79.3 ms, sys: 30.3 ms, total: 110 ms
Wall time: 66.8 ms


,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Paris,43.689490,False,2026-02-01 21:00:35.978702,71,323,179.164861
4,5,Paris,91.719953,True,2023-04-02 21:00:35.978702,67,199,617.349590
5,6,Brazil,60.497154,False,2023-05-20 21:00:35.978702,53,57,489.821394
6,7,New York,41.789961,False,2024-12-14 21:00:35.978702,73,79,426.253303
7,8,Paris,27.220312,True,2024-07-13 21:00:35.978702,75,238,65.261914
...,...,...,...,...,...,...,...,...
499990,499991,Dubai,10.111300,True,2023-06-07 21:00:35.978702,62,250,495.787202
499991,499992,Moscow,14.317979,False,2024-04-08 21:00:35.978702,75,70,801.377008
499995,499996,Dubai,61.531318,False,2025-04-19 21:00:35.978702,67,448,157.442594
499996,499997,Berlin,74.712924,False,2025-10-25 21:00:35.978702,74,36,418.119068


## Step 3: Compare the number of rows returned and the execution times. Write a markdown cell explaining what predicate pushdown does and why it is fas

Both approaches return the same number of rows (users with age > 50), but the execution times are different. The PyArrow filtered read is usually faster because the filter is applied during the file read.

Predicate pushdown means that the filtering condition is pushed down to the storage layer. In Parquet, the engine can use metadata and column statistics (such as min and max values in row groups) to skip row groups that cannot satisfy the condition. As a result, less data is read from disk and processed, which makes the query faster compared to loading the entire dataset and filtering afterward in pandas.

In [311]:
pq.read_table('data.parquet', filters=[('age','>',50)]).num_rows

237812

In [312]:
df = pd.read_parquet('data.parquet')
len(df[df['age'] > 50])

237812

## Step 4: Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:

import duckdb
result = duckdb.sql("SELECT * FROM 'your_file.parquet' WHERE age > 50")

In [313]:
import duckdb
%time duckdb.sql("SELECT * FROM 'data.parquet' WHERE age > 50").df()


CPU times: user 83.9 ms, sys: 29.4 ms, total: 113 ms
Wall time: 123 ms


,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Paris,43.689490,False,2026-02-01 21:00:35.978702,71,323,179.164861
1,5,Paris,91.719953,True,2023-04-02 21:00:35.978702,67,199,617.349590
2,6,Brazil,60.497154,False,2023-05-20 21:00:35.978702,53,57,489.821394
3,7,New York,41.789961,False,2024-12-14 21:00:35.978702,73,79,426.253303
4,8,Paris,27.220312,True,2024-07-13 21:00:35.978702,75,238,65.261914
...,...,...,...,...,...,...,...,...
237807,499991,Dubai,10.111300,True,2023-06-07 21:00:35.978702,62,250,495.787202
237808,499992,Moscow,14.317979,False,2024-04-08 21:00:35.978702,75,70,801.377008
237809,499996,Dubai,61.531318,False,2025-04-19 21:00:35.978702,67,448,157.442594
237810,499997,Berlin,74.712924,False,2025-10-25 21:00:35.978702,74,36,418.119068


In [314]:
%time pq.read_table('data.parquet', filters=[('age','>',50)])

CPU times: user 86.2 ms, sys: 11.2 ms, total: 97.4 ms
Wall time: 40.7 ms


pyarrow.Table
user_id: int64
city: large_string
score: double
active: bool
signup_date: timestamp[us]
age: int64
sessions: int64
revenue: double
----
user_id: [[1,5,6,7,8,...,131059,131067,131068,131071,131072],[131075,131078,131080,131081,131086,...,262137,262139,262140,262141,262144],[262148,262150,262151,262152,262155,...,393204,393206,393208,393212,393213],[393217,393218,393219,393220,393222,...,499991,499992,499996,499997,500000]]
city: [["Paris","Paris","Brazil","New York","Paris",...,"New York","Tbilisi","Tokyo","New York","Tokyo"],["Moscow","Baku","Brazil","Baku","Berlin",...,"Moscow","Berlin","London","Dubai","Tbilisi"],["Tbilisi","Tbilisi","London","London","Dubai",...,"New York","Tbilisi","Tbilisi","Tokyo","New York"],["New York","Baku","Baku","Tokyo","New York",...,"Dubai","Moscow","Dubai","Berlin","London"]]
score: [[43.689490033566045,91.71995305366835,60.49715376221981,41.78996051330323,27.220311643727456,...,69.69202532730313,20.260574805237873,73.70424820759605,46.3266

In [315]:
%%time

df = pd.read_parquet('data.parquet')
df[df['age'] > 50]

CPU times: user 83.4 ms, sys: 27.6 ms, total: 111 ms
Wall time: 67.2 ms


,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Paris,43.689490,False,2026-02-01 21:00:35.978702,71,323,179.164861
4,5,Paris,91.719953,True,2023-04-02 21:00:35.978702,67,199,617.349590
5,6,Brazil,60.497154,False,2023-05-20 21:00:35.978702,53,57,489.821394
6,7,New York,41.789961,False,2024-12-14 21:00:35.978702,73,79,426.253303
7,8,Paris,27.220312,True,2024-07-13 21:00:35.978702,75,238,65.261914
...,...,...,...,...,...,...,...,...
499990,499991,Dubai,10.111300,True,2023-06-07 21:00:35.978702,62,250,495.787202
499991,499992,Moscow,14.317979,False,2024-04-08 21:00:35.978702,75,70,801.377008
499995,499996,Dubai,61.531318,False,2025-04-19 21:00:35.978702,67,448,157.442594
499996,499997,Berlin,74.712924,False,2025-10-25 21:00:35.978702,74,36,418.119068


The filtered query was executed using three approaches: DuckDB, PyArrow with filters, and pandas filtering after loading the full dataset. All methods returned the same number of rows, but the execution times were slightly different.

PyArrow was the fastest because it applies predicate pushdown directly when reading the Parquet file, allowing it to skip row groups that do not match the condition. DuckDB also performs predicate pushdown and queries the Parquet file directly using SQL, but converting the result to a pandas DataFrame (`.df()`) can add some overhead. The pandas approach is usually slower because it first loads the entire dataset into memory and only then applies the filter.

# Task 4: pandas vs DuckDB — Identical Queries

Run the same five analytical queries in both pandas and DuckDB on the dataset from Task 1. For each query, record the code and the execution time.

## Query 1: Count records per city.



In [316]:
data

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Paris,43.689490,False,2026-02-01 21:00:35.978702,71,323,179.164861
1,2,Dubai,97.403462,True,2025-06-02 21:00:35.978702,39,352,614.893856
2,3,London,66.148238,False,2025-11-08 21:00:35.978702,23,279,217.069799
3,4,Baku,21.993544,False,2025-05-22 21:00:35.978702,20,129,195.459273
4,5,Paris,91.719953,True,2023-04-02 21:00:35.978702,67,199,617.349590
...,...,...,...,...,...,...,...,...
499995,499996,Dubai,61.531318,False,2025-04-19 21:00:35.978702,67,448,157.442594
499996,499997,Berlin,74.712924,False,2025-10-25 21:00:35.978702,74,36,418.119068
499997,499998,Moscow,48.430697,True,2023-09-03 21:00:35.978702,27,205,803.122069
499998,499999,London,89.680757,True,2026-01-24 21:00:35.978702,50,432,602.050097


In [317]:
%%time
    
data.groupby('city')['user_id'].count()

CPU times: user 24.4 ms, sys: 6.84 ms, total: 31.2 ms
Wall time: 29.7 ms


city
Baku        50216
Berlin      50059
Brazil      50061
Dubai       49931
London      49885
Moscow      49724
New York    50113
Paris       49982
Tbilisi     50059
Tokyo       49970
Name: user_id, dtype: int64

In [318]:
%%time
    
duckdb.sql(""" SELECT city, count(user_id)
FROM data
GROUP BY city """)
           

CPU times: user 44.3 ms, sys: 10 ms, total: 54.3 ms
Wall time: 54.4 ms


┌──────────┬────────────────┐
│   city   │ count(user_id) │
│ varchar  │     int64      │
├──────────┼────────────────┤
│ Brazil   │          50061 │
│ Baku     │          50216 │
│ Tbilisi  │          50059 │
│ London   │          49885 │
│ Berlin   │          50059 │
│ Moscow   │          49724 │
│ New York │          50113 │
│ Paris    │          49982 │
│ Tokyo    │          49970 │
│ Dubai    │          49931 │
├──────────┴────────────────┤
│ 10 rows         2 columns │
└───────────────────────────┘

## Query 2: Average score by city, ordered by average score descending.



In [319]:
%time data.groupby('city')['score'].mean().sort_values(ascending=False)

CPU times: user 25.5 ms, sys: 4.48 ms, total: 30 ms
Wall time: 27.2 ms


city
Tbilisi     50.311448
Baku        50.151899
Tokyo       50.135854
Paris       50.075511
London      50.063780
Berlin      50.013372
Dubai       50.009568
Moscow      49.996347
Brazil      49.994487
New York    49.879502
Name: score, dtype: float64

In [320]:
%%time 
duckdb.sql("""SELECT city, AVG(score)
FROM data
GROUP BY city
ORDER BY AVG(score) DESC""")

CPU times: user 41.5 ms, sys: 8.83 ms, total: 50.4 ms
Wall time: 48.9 ms


┌──────────┬────────────────────┐
│   city   │     avg(score)     │
│ varchar  │       double       │
├──────────┼────────────────────┤
│ Tbilisi  │ 50.311447639720384 │
│ Baku     │  50.15189868104277 │
│ Tokyo    │ 50.135854072614144 │
│ Paris    │  50.07551149964857 │
│ London   │  50.06377999045983 │
│ Berlin   │  50.01337161408978 │
│ Dubai    │  50.00956774673048 │
│ Moscow   │  49.99634719162364 │
│ Brazil   │ 49.994487181000736 │
│ New York │  49.87950214845574 │
├──────────┴────────────────────┤
│ 10 rows             2 columns │
└───────────────────────────────┘

## Query 3: For each city, find the percentage of active users whose score is above 75.


In [321]:
%%time
q3_pandas = (
    data.loc[data["active"]]
        .assign(high=lambda x: x["score"] > 75)
        .groupby("city")["high"]
        .mean()
        .mul(100)
)

q3_pandas

CPU times: user 35.3 ms, sys: 9.82 ms, total: 45.2 ms
Wall time: 45.5 ms


city
Baku        25.101361
Berlin      24.981993
Brazil      24.913233
Dubai       25.431900
London      24.984980
Moscow      25.054554
New York    24.764614
Paris       25.064785
Tbilisi     25.538499
Tokyo       25.355897
Name: high, dtype: float64

In [322]:
%%time
q3_duckdb = duckdb.sql("""
SELECT 
    city,
    100.0 * AVG(CASE WHEN score > 75 THEN 1 ELSE 0 END) AS pct_active_score_gt_75
FROM data
WHERE active = TRUE
GROUP BY city
ORDER BY city
""")

q3_duckdb

CPU times: user 40.8 ms, sys: 7.72 ms, total: 48.5 ms
Wall time: 46.6 ms


┌──────────┬────────────────────────┐
│   city   │ pct_active_score_gt_75 │
│ varchar  │         double         │
├──────────┼────────────────────────┤
│ Baku     │     25.101360844606802 │
│ Berlin   │      24.98199279711885 │
│ Brazil   │      24.91323253680137 │
│ Dubai    │     25.431900361590998 │
│ London   │     24.984980173829456 │
│ Moscow   │       25.0545542713974 │
│ New York │     24.764613966905724 │
│ Paris    │     25.064784914085237 │
│ Tbilisi  │     25.538498633660183 │
│ Tokyo    │      25.35589686008742 │
├──────────┴────────────────────────┤
│ 10 rows                 2 columns │
└───────────────────────────────────┘

## Query 4: Find the top 10 users by score for each city (window function / rank).


In [323]:
%%time
q4_pandas = (
    data.assign(rank=data.groupby("city")["score"].rank(method="first", ascending=False))
        .query("rank <= 10")
        .sort_values(["city","rank"])
)
q4_pandas

CPU times: user 360 ms, sys: 18.8 ms, total: 379 ms
Wall time: 384 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,rank
260670,260671,Baku,99.999493,False,2023-09-23 21:00:35.978702,70,55,136.141516,1.0
159525,159526,Baku,99.996430,False,2023-08-28 21:00:35.978702,78,272,544.755594,2.0
206508,206509,Baku,99.994056,True,2025-12-28 21:00:35.978702,21,135,361.665131,3.0
209113,209114,Baku,99.993964,True,2024-11-12 21:00:35.978702,23,33,88.518683,4.0
360623,360624,Baku,99.993005,True,2025-05-20 21:00:35.978702,26,406,4.828742,5.0
...,...,...,...,...,...,...,...,...,...
292755,292756,Tokyo,99.988919,True,2024-05-07 21:00:35.978702,67,476,495.721210,6.0
318828,318829,Tokyo,99.988068,True,2024-05-13 21:00:35.978702,59,259,95.062649,7.0
139162,139163,Tokyo,99.985818,True,2025-10-10 21:00:35.978702,79,296,103.359663,8.0
342482,342483,Tokyo,99.979608,False,2024-04-07 21:00:35.978702,71,310,407.642412,9.0


In [324]:
%%time
duckdb.sql("""
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY score DESC) AS rank
    FROM data
)
WHERE rank <= 10
""").df()

CPU times: user 793 ms, sys: 106 ms, total: 900 ms
Wall time: 270 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,rank
0,324298,Dubai,99.999003,True,2023-04-18 21:00:35.978702,45,115,874.249204,1
1,169257,Dubai,99.998943,True,2023-10-07 21:00:35.978702,22,1,377.561203,2
2,7101,Dubai,99.998401,False,2023-07-22 21:00:35.978702,33,252,574.876000,3
3,394008,Dubai,99.998235,False,2023-07-03 21:00:35.978702,72,320,376.483959,4
4,250235,Dubai,99.998183,False,2023-10-21 21:00:35.978702,77,179,329.384976,5
...,...,...,...,...,...,...,...,...,...
95,211842,New York,99.985983,False,2025-08-22 21:00:35.978702,30,454,401.990962,6
96,62709,New York,99.985601,False,2024-08-20 21:00:35.978702,31,79,998.743226,7
97,174774,New York,99.984856,True,2025-12-09 21:00:35.978702,74,125,346.439924,8
98,42407,New York,99.983991,False,2023-08-21 21:00:35.978702,25,390,367.026701,9


## Query 5: Compute a running total of scores ordered by user_id, partitioned by city.



In [325]:
%%time
q5_pandas = data.sort_values(["city","user_id"])
q5_pandas["running_total"] = q5_pandas.groupby("city")["score"].cumsum().round(2)

CPU times: user 122 ms, sys: 40.1 ms, total: 163 ms
Wall time: 164 ms


In [326]:
q5_pandas

,user_id,city,score,active,signup_date,age,sessions,revenue,running_total
3,4,Baku,21.993544,False,2025-05-22 21:00:35.978702,20,129,195.459273,21.99
9,10,Baku,39.357197,True,2025-03-03 21:00:35.978702,45,242,522.293514,61.35
15,16,Baku,67.966054,False,2025-08-28 21:00:35.978702,44,314,255.933348,129.32
20,21,Baku,53.725081,True,2024-11-23 21:00:35.978702,64,170,951.105930,183.04
32,33,Baku,58.181235,True,2025-04-12 21:00:35.978702,40,346,566.704058,241.22
...,...,...,...,...,...,...,...,...,...
499960,499961,Tokyo,71.050809,True,2024-10-19 21:00:35.978702,36,498,77.928552,2505088.16
499965,499966,Tokyo,77.613238,False,2023-06-15 21:00:35.978702,56,354,337.266743,2505165.78
499970,499971,Tokyo,57.251573,True,2025-02-13 21:00:35.978702,47,137,327.412880,2505223.03
499972,499973,Tokyo,41.334913,False,2024-07-18 21:00:35.978702,18,419,116.494284,2505264.36


In [327]:
%%time
duckdb.sql("""
SELECT *,
SUM(score) OVER (PARTITION BY city ORDER BY user_id) AS running_total
FROM data
""").df()

CPU times: user 816 ms, sys: 196 ms, total: 1.01 s
Wall time: 607 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,running_total
0,3,London,66.148238,False,2025-11-08 21:00:35.978702,23,279,217.069799,6.614824e+01
1,9,London,9.353333,True,2024-07-15 21:00:35.978702,35,169,273.116338,7.550157e+01
2,12,London,77.267889,True,2025-10-02 21:00:35.978702,54,1,85.207302,1.527695e+02
3,13,London,61.616970,True,2024-05-09 21:00:35.978702,39,34,220.527548,2.143864e+02
4,18,London,8.859391,False,2025-01-26 21:00:35.978702,35,47,649.863846,2.232458e+02
...,...,...,...,...,...,...,...,...,...
499995,428873,Brazil,60.830635,False,2023-12-24 21:00:35.978702,38,7,73.696983,2.146974e+06
499996,428877,Brazil,3.977582,True,2025-08-30 21:00:35.978702,47,480,75.883949,2.146978e+06
499997,428879,Brazil,5.899681,True,2023-07-26 21:00:35.978702,25,457,723.188507,2.146983e+06
499998,428904,Brazil,76.372420,True,2025-12-05 21:00:35.978702,28,225,49.933566,2.147060e+06


## Step 5: Write a markdown cell comparing:

Which tool was easier to express each query in?
Which was faster?
For which queries did the difference matter most?


### Comparison of Pandas and DuckDB Queries

In this analysis, the same five analytical queries were implemented using both pandas and DuckDB.

**Ease of expression:**  
For simple aggregations such as counting users by city or computing average scores, pandas was slightly easier and more concise because it uses direct DataFrame operations like `groupby()` and `mean()`. However, for more complex operations such as ranking users within each city or computing running totals, DuckDB's SQL syntax was clearer because window functions like `ROW_NUMBER()` and `SUM() OVER()` express the logic more directly.

**Performance:**  
Based on the execution times shown above, pandas was generally faster in this experiment. This is likely because the dataset was already loaded into a pandas DataFrame in memory, allowing pandas to apply optimized vectorized operations such as `groupby`, `cumsum`, and `rank`. DuckDB queries required additional overhead to execute the SQL engine and convert the result back into a pandas DataFrame using `.df()`.

**Where the difference mattered most:**  
The performance difference was most noticeable for the window-function queries (top 10 ranking and running totals). These queries were significantly slower in DuckDB compared to pandas. For simpler aggregation queries, such as counting users or computing averages by city, the performance difference between the two tools was relatively small.

Overall, pandas performed better for in-memory DataFrame operations in this dataset, while DuckDB provided a clearer and more expressive SQL interface for complex analytical queries.

# Task 5: Arrow as the Bridge


## Step 1: Create a pandas DataFrame and convert it to an Arrow Table using pyarrow.Table.from_pandas().

In [328]:
import pandas as pd

df0 = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "city": ["Baku", "Berlin", "Tokyo", "Paris"],
    "score": [78.5, 91.2, 65.4, 88.7],
    "active": [True, False, True, True]
})

df0

,user_id,city,score,active
0,1,Baku,78.5,True
1,2,Berlin,91.2,False
2,3,Tokyo,65.4,True
3,4,Paris,88.7,True


In [329]:
import pyarrow as pa

p = pa.Table.from_pandas(df0)
p

pyarrow.Table
user_id: int64
city: large_string
score: double
active: bool
----
user_id: [[1,2,3,4]]
city: [["Baku","Berlin","Tokyo","Paris"]]
score: [[78.5,91.2,65.4,88.7]]
active: [[true,false,true,true]]

## Step 2: Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.



In [330]:
df0.dtypes

user_id      int64
city           str
score      float64
active        bool
dtype: object

In [331]:
p.schema

user_id: int64
city: large_string
score: double
active: bool
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 707

The Arrow schema is mostly similar to the pandas dtypes, but there are a few differences.

In pandas, the `city` column has dtype `object` (string stored as Python objects), while in Arrow it is stored as `large_string`, which is a dedicated columnar string type. This makes Arrow more memory-efficient and faster for analytics.

Also, pandas `float64` corresponds to Arrow `double`, which represent the same numeric type. The integer and boolean types (`int64` and `bool`) remain the same in both systems.

## Step 3: Write the Arrow Table to Parquet using pyarrow.parquet.write_table(). Read it back into a new Arrow Table.

In [332]:
pq.write_table(p, "arrow_data.parquet")

In [333]:
p2 = pq.read_table("arrow_data.parquet")
p2

pyarrow.Table
user_id: int64
city: large_string
score: double
active: bool
----
user_id: [[1,2,3,4]]
city: [["Baku","Berlin","Tokyo","Paris"]]
score: [[78.5,91.2,65.4,88.7]]
active: [[true,false,true,true]]

## Step 4: Convert the Arrow Table back to a pandas DataFrame using .to_pandas(). Verify the data matches the original.

In [334]:
df3 =p.to_pandas()
df3

,user_id,city,score,active
0,1,Baku,78.5,True
1,2,Berlin,91.2,False
2,3,Tokyo,65.4,True
3,4,Paris,88.7,True


In [335]:
df3.equals(df0)

True

## Step 5: Demonstrate the pandas dtype_backend="pyarrow" option
by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.



In [336]:
df4= pd.read_parquet('arrow_data.parquet', dtype_backend= 'pyarrow')
df4

,user_id,city,score,active
0,1,Baku,78.5,True
1,2,Berlin,91.2,False
2,3,Tokyo,65.4,True
3,4,Paris,88.7,True


In [337]:
df4.dtypes

user_id           int64[pyarrow]
city       large_string[pyarrow]
score            double[pyarrow]
active             bool[pyarrow]
dtype: object

With `dtype_backend="pyarrow"`, pandas keeps columns as Arrow-backed dtypes instead of converting them to NumPy/object types.

- `city` becomes `large_string[pyarrow]` (Arrow string) instead of pandas `object` (Python strings).
- Numeric columns remain 64-bit but show as Arrow types: `int64[pyarrow]`, `double[pyarrow]`.
- `active` becomes `bool[pyarrow]`.

This matters because Arrow-backed dtypes can reduce Python-object overhead (especially for strings) and can be more memory-efficient while staying compatible with Arrow/Parquet workflows.

## Step 6: Write a markdown cell explaining the role of Arrow in the modern analytics stack. How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?



Arrow acts as a common in-memory data format in the modern analytics stack. It provides a fast, columnar representation of data that can be shared between different tools without expensive conversions.

Parquet stores data on disk in a compressed columnar format. When the data is read, it can be loaded into memory as an Arrow Table. Libraries like pandas and DuckDB can then directly work with Arrow data.

This allows efficient interoperability: Parquet handles storage, Arrow provides a fast in-memory structure, pandas is used for data analysis in Python, and DuckDB performs SQL queries on the same data without unnecessary copying.